# MO-IT148 — Week 6: Data Retrieval and Processing
**H3101 Group ADET**

This notebook retrieves the IoT logistics records stored on the blockchain (Milestone 1 output), structures them into a DataFrame, extracts numeric values, handles missing data, and saves the cleaned dataset to `cleaned_iot_data.csv`.

> **Prerequisite:** Run this in the SAME session/kernel right after your Milestone 1 notebook, so that `web3` and `contract` are already defined. If running fresh, run the *Reconnect* cell below first.

## (Optional) Reconnect — run this only if starting a fresh kernel

In [1]:
# Run this ONLY if `contract` is not already defined (fresh kernel / new session).
# If you just ran Milestone 1 in the same kernel, you can SKIP this cell.

from web3 import Web3

ganache_url = "http://127.0.0.1:7545"   # change to 8545 if your Ganache uses that port
web3 = Web3(Web3.HTTPProvider(ganache_url))
print("Connected:", web3.is_connected())

# Paste the deployed contract address from Milestone 1 (Remix or py-solc-x output):
contract_address = "0xc10a55212cc6770924D1268396acDFb56A58423F"

abi = [
    {"inputs": [{"internalType": "string", "name": "_deviceId", "type": "string"},
                {"internalType": "string", "name": "_dataType", "type": "string"},
                {"internalType": "string", "name": "_value", "type": "string"}],
     "name": "storeData", "outputs": [], "stateMutability": "nonpayable", "type": "function"},
    {"inputs": [{"internalType": "uint256", "name": "index", "type": "uint256"}],
     "name": "getRecord",
     "outputs": [{"internalType": "string", "name": "", "type": "string"},
                 {"internalType": "string", "name": "", "type": "string"},
                 {"internalType": "string", "name": "", "type": "string"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "getTotalRecords",
     "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
     "stateMutability": "view", "type": "function"},
]

contract = web3.eth.contract(address=Web3.to_checksum_address(contract_address), abi=abi)
print("Contract reconnected.")


Connected: True
Contract reconnected.


## Step 1 — Get the Total Number of Stored Records

In [2]:
total_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {total_records}")


Total IoT records stored: 100


## Step 2 — Fetch All Stored IoT Data into a DataFrame

Note: our `TrackingSystem` contract stores **3 fields** per record — `deviceId`, `dataType`, and `value`.
The `value` field is a combined string (e.g. `PKG:1064 | LOC:.. | TEMP:..C | STATUS:.. | TS:..`),
so we also parse it back into separate columns for analysis.

In [3]:
import pandas as pd

# Retrieve all IoT records (contract returns 3 strings: deviceId, dataType, value)
data = []
for i in range(total_records):
    record = contract.functions.getRecord(i).call()
    data.append({
        "device_id":  record[0],   # e.g. RFID_0
        "data_type":  record[1],   # e.g. Logistics
        "data_value": record[2],   # combined string
    })

df = pd.DataFrame(data)

# --- Parse the combined data_value string back into separate columns ---
df["package_id"]  = df["data_value"].str.extract(r"PKG:([^|]+)").iloc[:, 0].str.strip()
df["location"]    = df["data_value"].str.extract(r"LOC:([^|]+)").iloc[:, 0].str.strip()
df["temperature"] = df["data_value"].str.extract(r"TEMP:([0-9.]+)").iloc[:, 0]
df["status"]      = df["data_value"].str.extract(r"STATUS:([^|]+)").iloc[:, 0].str.strip()
df["timestamp"]   = df["data_value"].str.extract(r"TS:(.+)$").iloc[:, 0].str.strip()

# Convert timestamp text to a proper datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

print("Retrieved and structured records:")
df.head()


Retrieved and structured records:


,device_id,data_type,data_value,package_id,location,temperature,status,timestamp
0,RFID_0,Logistics,"PKG:1064 | LOC:12.422315329212937,121.19571255...",1064,"12.422315329212937,121.19571255545216",3.8241911589108772,In Transit,2026-01-01 00:00:00
1,RFID_1,Logistics,"PKG:1060 | LOC:13.513377156951517,122.51412936...",1060,"13.513377156951517,122.51412936901205",2.9270166180818755,In Transit,2026-01-01 01:00:00
2,RFID_2,Logistics,"PKG:1015 | LOC:14.291035309309276,122.54491741...",1015,"14.291035309309276,122.54491741023237",5.4660655602994845,In Transit,2026-01-01 02:00:00
3,RFID_3,Logistics,"PKG:1050 | LOC:10.539635016080613,123.66811295...",1050,"10.539635016080613,123.66811295673534",4.4416599739461065,Delayed,2026-01-01 03:00:00
4,RFID_4,Logistics,"PKG:1048 | LOC:10.465063725075503,122.47830568...",1048,"10.465063725075503,122.47830568516888",2.5822828478501743,In Transit,2026-01-01 04:00:00


## Step 3 — Preprocess: Extract Numeric Values & Handle Missing Data

Some IoT readings contain units or text (e.g. `"22.5°C"`). Here we extract the numeric temperature value.
We then check for missing values and fill them: minor gaps with 0, or you may switch to mean/median if significant.

In [4]:
import numpy as np

# Extract numeric temperature value (already isolated above, but ensure it's float)
df["numeric_value"] = pd.to_numeric(df["temperature"], errors="coerce")

# Also make package_id numeric where possible
df["package_id"] = pd.to_numeric(df["package_id"], errors="coerce")

# --- Identify missing values ---
print("Missing values per column:")
print(df.isnull().sum())

# --- Handle missing values ---
# Minor missing -> fill with 0. (Switch to mean/median if a column has many gaps.)
# Example for mean instead: df["numeric_value"].fillna(df["numeric_value"].mean(), inplace=True)
df.fillna(0, inplace=True)

print("\nCleaned data:")
df.head()


Missing values per column:
device_id        0
data_type        0
data_value       0
package_id       0
location         0
temperature      0
status           0
timestamp        0
numeric_value    0
dtype: int64

Cleaned data:


,device_id,data_type,data_value,package_id,location,temperature,status,timestamp,numeric_value
0,RFID_0,Logistics,"PKG:1064 | LOC:12.422315329212937,121.19571255...",1064,"12.422315329212937,121.19571255545216",3.8241911589108772,In Transit,2026-01-01 00:00:00,3.824191
1,RFID_1,Logistics,"PKG:1060 | LOC:13.513377156951517,122.51412936...",1060,"13.513377156951517,122.51412936901205",2.9270166180818755,In Transit,2026-01-01 01:00:00,2.927017
2,RFID_2,Logistics,"PKG:1015 | LOC:14.291035309309276,122.54491741...",1015,"14.291035309309276,122.54491741023237",5.4660655602994845,In Transit,2026-01-01 02:00:00,5.466066
3,RFID_3,Logistics,"PKG:1050 | LOC:10.539635016080613,123.66811295...",1050,"10.539635016080613,123.66811295673534",4.4416599739461065,Delayed,2026-01-01 03:00:00,4.441660
4,RFID_4,Logistics,"PKG:1048 | LOC:10.465063725075503,122.47830568...",1048,"10.465063725075503,122.47830568516888",2.5822828478501743,In Transit,2026-01-01 04:00:00,2.582283


## Step 4 — Save the Cleaned Data to CSV

In [5]:
# Save cleaned IoT data to a CSV file
df.to_csv("cleaned_iot_data.csv", index=False)
print("Cleaned IoT data saved successfully as cleaned_iot_data.csv")

# Preview the saved file
print(f"\nRows: {len(df)} | Columns: {list(df.columns)}")
df.head(10)


Cleaned IoT data saved successfully as cleaned_iot_data.csv

Rows: 100 | Columns: ['device_id', 'data_type', 'data_value', 'package_id', 'location', 'temperature', 'status', 'timestamp', 'numeric_value']


,device_id,data_type,data_value,package_id,location,temperature,status,timestamp,numeric_value
0,RFID_0,Logistics,"PKG:1064 | LOC:12.422315329212937,121.19571255...",1064,"12.422315329212937,121.19571255545216",3.8241911589108772,In Transit,2026-01-01 00:00:00,3.824191
1,RFID_1,Logistics,"PKG:1060 | LOC:13.513377156951517,122.51412936...",1060,"13.513377156951517,122.51412936901205",2.9270166180818755,In Transit,2026-01-01 01:00:00,2.927017
2,RFID_2,Logistics,"PKG:1015 | LOC:14.291035309309276,122.54491741...",1015,"14.291035309309276,122.54491741023237",5.4660655602994845,In Transit,2026-01-01 02:00:00,5.466066
3,RFID_3,Logistics,"PKG:1050 | LOC:10.539635016080613,123.66811295...",1050,"10.539635016080613,123.66811295673534",4.4416599739461065,Delayed,2026-01-01 03:00:00,4.441660
4,RFID_4,Logistics,"PKG:1048 | LOC:10.465063725075503,122.47830568...",1048,"10.465063725075503,122.47830568516888",2.5822828478501743,In Transit,2026-01-01 04:00:00,2.582283
5,RFID_5,Logistics,"PKG:1030 | LOC:11.355078993069949,123.43437373...",1030,"11.355078993069949,123.43437373110355",6.2056764810621186,Delayed,2026-01-01 05:00:00,6.205676
6,RFID_6,Logistics,"PKG:1025 | LOC:10.877796610972517,123.46294021...",1025,"10.877796610972517,123.46294021019365",7.774785607097465,Delivered,2026-01-01 06:00:00,7.774786
7,RFID_7,Logistics,"PKG:1050 | LOC:13.457388005106424,124.96562700...",1050,"13.457388005106424,124.96562700704685",5.191734199473666,In Transit,2026-01-01 07:00:00,5.191734
8,RFID_8,Logistics,"PKG:1013 | LOC:14.627323841226758,124.18886582...",1013,"14.627323841226758,124.18886582305335",5.340050188455926,Delayed,2026-01-01 08:00:00,5.340050
9,RFID_9,Logistics,"PKG:1069 | LOC:12.936688144706656,124.00181061...",1069,"12.936688144706656,124.0018106118917",6.663358817400176,Delayed,2026-01-01 09:00:00,6.663359
